In [1]:
# 1. Import the necessary libraries
import pandas as pd

# 2. Load your Excel files (ensure they're in the same folder as your notebook or provide full path)
train_df = pd.read_csv("train_dataset.csv")
test_df = pd.read_excel("test_dataset.xlsx")

# 3. Save them as .parquet
train_df.to_parquet("train.parquet", index=False)
test_df.to_parquet("test.parquet", index=False)

print("Parquet files created successfully!")

Parquet files created successfully!


In [2]:
from huggingface_hub import login
import os

login(token=os.environ['HF_TOKEN'])


c:\Users\radha\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from huggingface_hub import create_repo, upload_file

# 1. Create the dataset repo (change your_username and dataset name!)
repo_id = "agi123/optima_restore_ha"
create_repo(repo_id, repo_type="dataset")

# 2. Upload the parquet files
upload_file(path_or_fileobj="train.parquet", path_in_repo="data/train.parquet", repo_id=repo_id, repo_type="dataset")
upload_file(path_or_fileobj="test.parquet", path_in_repo="data/test.parquet", repo_id=repo_id, repo_type="dataset")

print("Files uploaded successfully!")


train.parquet: 100%|██████████| 276k/276k [00:01<00:00, 264kB/s]
test.parquet: 100%|██████████| 9.87k/9.87k [00:00<00:00, 33.1kB/s]


Files uploaded successfully!


In [4]:
yaml_config = """
dataset_info:
  features:
    - name: question
      dtype: string
    - name: answer
      dtype: string
  splits:
    - name: train
      files:
        - path: data/train.parquet
    - name: test
      files:
        - path: data/test.parquet
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_config)

# Upload it to Hugging Face
upload_file(path_or_fileobj="dataset.yaml", path_in_repo="dataset.yaml", repo_id=repo_id, repo_type="dataset")

print("YAML file uploaded!")


YAML file uploaded!


In [9]:
%pip install wandb

Note: you may need to restart the kernel to use updated packages.


# 2. Pushing manual test data and results into wandb

In [17]:
import wandb
wandb.login()  # Make sure this uses the correct account

True

In [18]:
import wandb

run = wandb.init(  
    project="finetune_Meta-llama-3-8B_20250726",
    id="3kzdk0d6",
    resume="allow"
)

print("✅ Resumed run ID:", run.id)


✅ Resumed run ID: 3kzdk0d6


In [21]:
artifact_name = 'sft_run_llama3-8B_ep5_bs3_00_config_and_results'

In [ ]:
#3. Get the existing artifact (This downloads all files in the artifact to a temp folder (like ./artifacts/artifact_name/v0).)
artifact = run.use_artifact(f"{artifact_name}:v0", type="training-config")
artifact_dir = artifact.download()

wandb:   3 of 3 files downloaded.  


In [24]:
artifact_dir

'd:\\findTuning\\rk\\rk\\artifacts\\sft_run_llama3-8B_ep5_bs3_00_config_and_results-v1'

In [25]:
#4. Read the evaluation results.
import pandas as pd

df = pd.read_excel("accuracyCheck_llama3-8b_ep6_bs3_nn0.005_ls0.1.xlsx")  # Your file here

In [26]:
#5. Calculate total_score and score_percent
total_score = df["score"].sum()
score_percent = (total_score/30)*100

In [27]:
total_score

np.float64(18.0)

In [28]:
score_percent

np.float64(60.0)

In [29]:
#6. Convert the evalutaion results file
df.to_csv("eval_results.csv", index=False)  # or use .parquet if you want

In [30]:
#7. Log total_score and score_percent into current run
run.log({
    "total_score": total_score,
    "score_percent": score_percent
})

In [31]:
# 7. Add your evaluation file to the artifact folder
import shutil

# Assuming you already saved it
shutil.copy("eval_results.csv", artifact_dir)

'd:\\findTuning\\rk\\rk\\artifacts\\sft_run_llama3-8B_ep5_bs3_00_config_and_results-v1\\eval_results.csv'

In [32]:
#8. Create a new version of the artifact with the new file included
new_artifact = wandb.Artifact(
    name=f"{artifact_name}",  # ✅ valid name
    type="training-config",
    description="Updated with eval results"
)


# Add ALL files from previous artifact + new file
new_artifact.add_dir(artifact_dir)

run.log_artifact(new_artifact)


wandb: Adding directory to artifact (d:\findTuning\rk\rk\artifacts\sft_run_llama3-8B_ep5_bs3_00_config_and_results-v1)... Done. 0.1s


<Artifact sft_run_llama3-8B_ep5_bs3_00_config_and_results>

In [33]:
#9. Log the DataFrame as a W&B table
table = wandb.Table(dataframe=df)
run.log({"eval_results_table": table})

run.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


score_percent,▁
total_score,▁
eval/loss,4.54422
eval/runtime,0.8488
eval/samples_per_second,35.346
eval/steps_per_second,11.782
score_percent,60
total_score,18
train/epoch,6
train/global_step,4230
train/grad_norm,2.60938


In [34]:
df

,question,answer,adapter_model_answer,score
0,What is HDFC ERGO Optima Restore?,Optima Restore is a health insurance plan from...,HDFC ERGO Optima Restore is a health insurance...,0.5
1,"According to HDFC ERGO Optima Restore, will I ...","Yes, preventive health check-ups are reimburse...","Yes, under the HDFC ERGO Optima Restore policy...",1.0
2,"According to HDFC ERGO Optima Restore, lets sa...","Yes, in emergencies where ground transport is ...","Yes, in case of a very emergency medical situa...",1.0
3,"According to HDFC ERGO Optima Restore, After d...",Post-hospitalization costs for things like fol...,"Yes, the HDFC ERGO Optima Restore policy will ...",0.5
4,"According to HDFC ERGO Optima Restore, If I go...","No, coverage is limited to treatments within I...","No, according to HDFC ERGO Optima Restore, if ...",0.0
5,"According to HDFC ERGO Optima Restore, self-ha...","War injuries, self-harm, drug abuse, infertili...","No, self-harm and drug abuse are not covered u...",1.0
6,"According to HDFC ERGO Optima Restore, If I wi...","Yes, that is called portability. You can carry...","Yes, under the HDFC ERGO Optima Restore policy...",1.0
7,"According to HDFC ERGO Optima Restore, If I go...",E-Opinion refers to a second medical opinion s...,"Yes, if you get diagnosed with a critical illn...",1.0
8,"According to HDFC ERGO Optima Restore, If I ge...",Yes! The policy covers single private AC rooms...,"Yes, the HDFC ERGO Optima Restore policy will ...",1.0
9,"According to HDFC ERGO Optima Restore, Can I c...","Yes, for specific day care procedures like cat...",According to the HDFC ERGO Optima Restore poli...,1.0
